# GPU probe: `probe` on a CUDA device

## Question and why it matters
Can the growth/shape `probe` (width, area, edge, gyration windows per ant) run on the GPU for thousands of (rule, seed) runs, with output **byte-identical** to the CPU `probe`, and how much faster is it? Links: `README.md`, `NOTEBOOK.md`.

## Assumptions and definitions
- CPU `probe` is the oracle; the GPU must equal it byte for byte (tested, held).
- Speedup = CPU whole-process wall / GPU whole-process wall, same 2400 specs, `probe 27 8192`, 6 CPU threads vs one RTX 4000 Ada at 2400 lanes.
- Assumed: machine idle during timing (checked by process list / load average in `bench.log`).

## Provenance
Prints the git commit, date, seed and a SHA-256 for every input file.

In [1]:
import hashlib, subprocess, datetime, json, re
from pathlib import Path
SEED = 0  # no randomness in this notebook: it only reads committed results
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").exists())
EXP = ROOT / "experiments" / "2026-10-16-gpu-probe"
R = EXP / "results"
inputs = [R/"bench.log", R/"bench_gpu1.json", R/"bench_gpu2.json", R/"spotcheck.txt", R/"spotcheck_specs.txt"]
missing = [p for p in inputs if not p.exists()]
assert not missing, missing
print("notebook run", datetime.date.today(), "seed", SEED)
print("HEAD", subprocess.run(["git","-C",str(ROOT),"rev-parse","--short","HEAD"],capture_output=True,text=True).stdout.strip())
for p in inputs: print(hashlib.sha256(p.read_bytes()).hexdigest()[:16], p.relative_to(ROOT))

notebook run 2026-10-17 seed 0
HEAD 53ccccc
1ff7b06d99ef84cc experiments/2026-10-16-gpu-probe/results/bench.log
cd692d366597c82c experiments/2026-10-16-gpu-probe/results/bench_gpu1.json
42713bbb43909ad6 experiments/2026-10-16-gpu-probe/results/bench_gpu2.json
254196fcde6ce8fc experiments/2026-10-16-gpu-probe/results/spotcheck.txt
088ea3b969c2352e experiments/2026-10-16-gpu-probe/results/spotcheck_specs.txt


## Data and method
`bash experiments/2026-10-16-gpu-probe/scripts/bench.sh` (WSL, repo root) times, sequentially, `gpu_probe 27 8192 --lanes 2400` (gpu1), `probe 27 8192` (cpu), and gpu2 on the 2400 kappa-leadrun-fresh specs. Heavy work is in the Rust binaries; this notebook parses `bench.log`.

In [2]:
log = (R/"bench.log").read_text()
wall = {m.group(1): float(m.group(2)) for m in re.finditer(r"(gpu1|cpu|gpu2) wall ([\d.]+) s", log)}
ident = re.findall(r"cmp vs dump: (\w+)", log)
print(wall, ident)
print(log.splitlines()[0])
for k in ("gpu1","gpu2"):
    s = json.loads((R/f"bench_{k}.json").read_text()); print(k, s)
print("speedup gpu1: %.2fx  gpu2: %.2fx" % (wall["cpu"]/wall["gpu1"], wall["cpu"]/wall["gpu2"]))

{'gpu1': 177.355451168, 'cpu': 681.648633111, 'gpu2': 176.949274154} ['identical', 'identical', 'identical']
git 53ccccc dirty=60 2026-10-17T17:38:27+13:00
gpu1 {'jobs': 2400, 'wall_secs': 160.9595, 'device': 'NVIDIA RTX 4000 Ada Generation', 'lanes': 2400, 'grid': 8192, 'slice': 1048576, 'max_log2': 27, 'pool_tiles': 2409257, 'launches': 128, 'oom_requeues': 0}
gpu2 {'jobs': 2400, 'wall_secs': 161.2787, 'device': 'NVIDIA RTX 4000 Ada Generation', 'lanes': 2400, 'grid': 8192, 'slice': 1048576, 'max_log2': 27, 'pool_tiles': 2409257, 'launches': 128, 'oom_requeues': 0}
speedup gpu1: 3.84x  gpu2: 3.85x


## Results
GPU 177.4 s / 176.9 s vs CPU 681.6 s: **3.84x**, all outputs byte-identical to the existing CPU dump (`oom_requeues` 0, pool 2.41e6 tiles).

*Shows:* the GPU port is exact on this workload and about 3.8x faster, inside the pre-registered 2x-6x (P2 held; P1, P3 held). *Does not show:* a general speed claim; n = 1 CPU run, one spec set, one GPU; lane scaling (P4) was not run.

## Caveats and controls
- The first GPU figure (177 s, 20:58 on 10-16) and the old CPU time (1231 s) were contaminated by other jobs; the clean CPU time is 681.6 s. A handover estimate of ~7x used the contaminated number and is superseded.
- Registered pool estimate (1.6e6 tiles) was about 50% low; no requeues.
- 24-spec spot check: `spotcheck.txt` (byte-identical).

## What this changes, next steps, open questions
Use `gpu_probe` for batches of hundreds+ plain-rule runs. Next: P4 (lanes 1024 to 2400), optional one-byte packing for n <= 32 if VRAM binds.